# Receipt VLM — merge a checkpoint & run it in the OCR pipeline

Takes **one training checkpoint** you produced (e.g. a `phase2_best.pt` / `phase2_epoch*.pt`
you downloaded from Kaggle), merges LoRA into the backbone to make an inference-ready
`receipt_vlm_500m_merged.pt`, and runs it through the real `receipt_ocr` pipeline on the
**same real-life labelled receipts the rest of the codebase evaluates on** — comparing each
prediction against its label and printing the standard acceptance metrics.

> **Run this on the project `.venv` kernel** (the one with the training deps installed). In
> VS Code: *Select Kernel* (top-right) → `dev_ocr/vlm_training/.venv`. Cell 1 warns if you're
> on the wrong interpreter. Then edit **cell 0** and *Run All*.

## 0. Configuration — edit then run

In [1]:
# Path to the checkpoint you downloaded from Kaggle (REQUIRED), e.g. a phase-2 snapshot.
# Cell 2 lists what's available if this path is wrong.
CHECKPOINT = r"D:\giorg\corsi\ESGI\5eme_annee\projet_annuel\price-tracker\dev_ocr\vlm_training\checkpoints\phase2_epoch04_loss0.3076.pt"

# Where to write the merged inference model. "" -> next to the checkpoint.
MERGED_OUT = ""

# Real-life labelled set (same images + labels used by scripts/evaluate.py).
# Leave IMAGES_DIR / LABELS_DIR = "" to use the repo's canonical locations.
IMAGES_DIR = ""
LABELS_DIR = ""
SPLIT = "test"            # "test" | "val" | "train" | "" for all labelled
REQUIRE_REVIEWED = False  # the project labels are pseudo-labels (not hand-reviewed yet), so
                          # keep False for now; set True once review_status.json marks them reviewed
MAX_IMAGES = 0            # 0 = every image in the split

## 1. Locate the packages (and check the interpreter)
Finds `dev_ocr/vlm_training`, puts `receipt_ocr` / `receipt_vlm` on the path, and **warns if
this kernel isn't the project `.venv`** — the merge and inference both need the training deps
(`transformers`, `tokenizers>=0.22`, …) that live there.

In [2]:
import os, sys
from pathlib import Path

def _find_train_pkg() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "export_checkpoint.py").is_file():
            return base
        cand = base / "dev_ocr" / "vlm_training"
        if (cand / "scripts" / "export_checkpoint.py").is_file():
            return cand
    raise RuntimeError("Couldn't locate dev_ocr/vlm_training -- set TRAIN_PKG manually")

TRAIN_PKG = _find_train_pkg()          # .../dev_ocr/vlm_training
DEV_OCR = TRAIN_PKG.parent             # .../dev_ocr
for p in (str(DEV_OCR / "src"), str(TRAIN_PKG)):   # receipt_ocr (src layout) + receipt_vlm
    if p not in sys.path:
        sys.path.insert(0, p)

# The project venv has the right deps; subprocesses use it, and we warn if THIS kernel isn't it.
_venv = TRAIN_PKG / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
VENV_PY = str(_venv) if _venv.exists() else sys.executable
on_venv = (not _venv.exists()) or Path(sys.executable).resolve() == _venv.resolve()
if not on_venv:
    print("!" * 78)
    print("This kernel is NOT the project venv:")
    print("   kernel :", sys.executable)
    print("   venv   :", VENV_PY)
    print("Cell 2 will still merge (it uses the venv), but cell 3 inference runs IN this kernel")
    print("and will fail on missing/old deps. Switch the kernel to .venv, then Run All.")
    print("!" * 78)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Train package:", TRAIN_PKG)
print("Device       :", DEVICE,
      "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only (slow generation)")

Train package: d:\giorg\corsi\ESGI\5eme_annee\projet_annuel\price-tracker\dev_ocr\vlm_training
Device       : cuda - NVIDIA GeForce RTX 2070


## 2. Merge the checkpoint
Runs `scripts/export_checkpoint.py` (on the venv interpreter) to fold the LoRA adapters into
the frozen backbone and write a single inference-ready `.pt`. Prints the script's own output
so any failure is visible, not hidden behind a `CalledProcessError`.

In [3]:
import subprocess
from pathlib import Path

ckpt = Path(CHECKPOINT)
if not ckpt.is_file():
    avail = sorted(p.name for p in ckpt.parent.glob("*.pt")) if ckpt.parent.is_dir() else []
    raise FileNotFoundError(
        f"CHECKPOINT not found: {ckpt.resolve()}\n"
        f"Available in {ckpt.parent}/: {avail or 'none -- download one from your Kaggle dataset'}"
    )
merged = Path(MERGED_OUT) if MERGED_OUT else ckpt.with_name("receipt_vlm_500m_merged.pt")

proc = subprocess.run(
    [VENV_PY, str(TRAIN_PKG / "scripts" / "export_checkpoint.py"),
     "--checkpoint", str(ckpt), "--output", str(merged)],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"export_checkpoint.py failed (exit {proc.returncode}) -- see error above")

MERGED = str(merged.resolve())
print("Merged ->", MERGED, f"({merged.stat().st_size/1e6:.0f} MB)")
print("Built from:", ckpt.name)

Merged checkpoint written: D:\giorg\corsi\ESGI\5eme_annee\projet_annuel\price-tracker\dev_ocr\vlm_training\checkpoints\receipt_vlm_500m_merged.pt (1818 MB)

Merged -> D:\giorg\corsi\ESGI\5eme_annee\projet_annuel\price-tracker\dev_ocr\vlm_training\checkpoints\receipt_vlm_500m_merged.pt (1818 MB)
Built from: phase2_epoch04_loss0.3076.pt


## 3. Run the OCR pipeline on the labelled real receipts
Loads the same real-life labelled receipts as `scripts/evaluate.py` (via `load_real_samples`),
runs each through `receipt_ocr.extract_receipt()` with the merged model as the VLM provider,
and prints the **prediction next to its ground-truth label** so you can eyeball quality.

> If you get *0 samples*, your labels aren't hand-reviewed yet — keep `REQUIRE_REVIEWED=False`
> in cell 0 (already the default).

In [4]:
import os, json, time
from pathlib import Path

os.environ.update({
    "RECEIPT_OCR_BACKEND":    "vlm",
    "RECEIPT_VLM_MODEL":      "receipt-vlm-500m",
    "RECEIPT_VLM_MODE":       "json",
    "RECEIPT_VLM_MODEL_PATH": MERGED,
})

from receipt_ocr import extract_receipt                       # full pipeline (VLM provider)
from receipt_vlm.data.real_photos import load_real_samples    # same loader evaluate.py uses
from receipt_vlm.data.schema import ticket_from_dict

images_dir = Path(IMAGES_DIR) if IMAGES_DIR else DEV_OCR / "data" / "raw" / "images_tickets_caisse"
labels_dir = Path(LABELS_DIR) if LABELS_DIR else TRAIN_PKG / "data" / "real_labels"
samples = load_real_samples(images_dir, labels_dir, split=(SPLIT or None),
                            require_reviewed=REQUIRE_REVIEWED)
if not samples:
    raise SystemExit(
        f"No labelled samples for split={SPLIT!r} (require_reviewed={REQUIRE_REVIEWED}) "
        f"under {labels_dir}. If the labels aren't hand-reviewed, set REQUIRE_REVIEWED=False."
    )
if MAX_IMAGES:
    samples = samples[:MAX_IMAGES]
print(f"Testing on {len(samples)} labelled '{SPLIT or 'all'}' receipts from {images_dir}\n")

predictions, golds, n_valid = [], [], 0
for s in samples:
    t = time.time()
    try:
        out = extract_receipt(str(s.image))
        pred = ticket_from_dict(out)
        n_valid += 1
    except Exception as e:
        out = {"error": f"{type(e).__name__}: {e}"}
        pred = ticket_from_dict({})
    predictions.append(pred)
    golds.append(s.ticket)
    print(f"=== {Path(s.image).name}  ({time.time()-t:.1f}s) ===")
    print("  predicted:", json.dumps(out, ensure_ascii=False)[:700])
    print("  gold     :", json.dumps(s.ticket.to_dict(), ensure_ascii=False)[:700], "\n")

print(f"Valid JSON: {n_valid}/{len(samples)}")

Testing on 5 labelled 'test' receipts from d:\giorg\corsi\ESGI\5eme_annee\projet_annuel\price-tracker\dev_ocr\data\raw\images_tickets_caisse



D:\Programmi\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


=== image_11.jpg  (70.5s) ===
  predicted: {"ticket": {"date": "20230622 16:34", "chaine_supermarche": "Parcours", "adresse": "", "produits": [{"nom_produit": "Poulet de Boeuf", "prix_unitaire_ou_kg": 1.08, "unites": 1}, {"nom_produit": "Poulet de Boeuf", "prix_unitaire_ou_kg": 1.06, "unites": 1}]}}
  gold     : {"ticket": {"date": "20260206 13:55", "chaine_supermarche": "MY MARKET", "adresse": "281 RUE DU FAUBOURG ST ANTOINE 75012 PARIS", "produits": [{"nom_produit": "LE QUARTIERE PATISSERIE BAGUETTE CUITE", "prix_unitaire_ou_kg": 1.3, "unites": 1}]}} 

=== image_13.jpg  (33.3s) ===
  predicted: {"ticket": {"date": "20230622 16:30", "chaine_supermarche": "Fran", "adresse": "", "produits": [{"nom_produit": "L'Eau de Poil", "prix_unitaire_ou_kg": 1.0, "unites": 1}, {"nom_produit": "L'Eau de Vie", "prix_unitaire_ou_kg": 1.0, "unites": 1}]}}
  gold     : {"ticket": {"date": "20240206 13:55", "chaine_supermarche": "My Daily", "adresse": "281 RUE DU FAUBOURG ST ANTOINE 75012 PARIS", "produi

## 4. Acceptance metrics (vs ground truth)
The same metrics `scripts/evaluate.py` reports — computed here on the pipeline's output so the
numbers reflect the *deployed* path, not just the raw model. (Tiny labelled set = directional
signal, not a firm number.)

In [5]:
from receipt_vlm.utils.metrics import evaluate_tickets

metrics = evaluate_tickets(predictions, golds)
TARGETS = [
    ("field_f1",       "Field F1",         "> 0.85"),
    ("product_recall", "Product recall",   "> 0.90"),
    ("price_mae",      "Price MAE (EUR)",  "< 0.05"),
    ("date_accuracy",  "Date exact match", "> 0.90"),
    ("anls",           "ANLS",             "> 0.80"),
]
print(f"{'Metric':22} {'Target':10} {'receipt-vlm-500m'}")
print("-" * 50)
for key, label, target in TARGETS:
    if key in metrics:
        print(f"{label:22} {target:10} {metrics[key]:.3f}")
print(f"\nn = {len(predictions)} labelled receipts  |  checkpoint: {Path(CHECKPOINT).name}")

Metric                 Target     receipt-vlm-500m
--------------------------------------------------
Field F1               > 0.85     0.000
Product recall         > 0.90     0.000
Price MAE (EUR)        < 0.05     0.000
Date exact match       > 0.90     0.000
ANLS                   > 0.80     0.185

n = 5 labelled receipts  |  checkpoint: phase2_epoch04_loss0.3076.pt


## 5. Deploy it
The merged model is a single `.pt`. To run it in your inference infrastructure (e.g. the OCR
worker), point the same env vars at it:

```bash
RECEIPT_OCR_BACKEND=vlm
RECEIPT_VLM_MODEL=receipt-vlm-500m
RECEIPT_VLM_MODE=json
RECEIPT_VLM_MODEL_PATH=/path/to/receipt_vlm_500m_merged.pt
```

PowerShell equivalent:
```powershell
$env:RECEIPT_OCR_BACKEND    = "vlm"
$env:RECEIPT_VLM_MODEL      = "receipt-vlm-500m"
$env:RECEIPT_VLM_MODE       = "json"
$env:RECEIPT_VLM_MODEL_PATH = "C:\path\to\receipt_vlm_500m_merged.pt"
```